In [2]:
import os
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

if "notebooks" in os.getcwd():
    os.chdir("..")

# ── Snowflake Connection ──
con = snowflake.connector.connect(
    user      = "Ruchita",
    password  = "RuchitaUday@0915",
    account   = "ZGQUOVJ-WT53494",
    warehouse = "COMPUTE_WH",
    database  = "RETAILPULSE_DB",
    schema    = "ANALYTICS"
)
cur = con.cursor()
print("Connected to Snowflake.")

# ── Load raw cleaned CSVs ──
BASE = r"C:\Users\HOLE\DATA ANALYST\RetailPulseAI - Project\01_Data\cleaned"

customers            = pd.read_csv(rf"{BASE}\customers_clean.csv")
products             = pd.read_csv(rf"{BASE}\products_clean.csv")
category_translation = pd.read_csv(rf"{BASE}\category_translation_clean.csv")
sellers              = pd.read_csv(rf"{BASE}\sellers_clean.csv")
orders               = pd.read_csv(rf"{BASE}\orders_clean.csv")
order_items          = pd.read_csv(rf"{BASE}\order_items_clean.csv")
payments             = pd.read_csv(rf"{BASE}\payments_clean.csv")
reviews              = pd.read_csv(rf"{BASE}\reviews_clean.csv")

orders["order_purchase_timestamp"]      = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
print("CSVs loaded.")


def load_table(df, table_name):
    """Truncate then reload a Snowflake table."""
    cur.execute(f"TRUNCATE TABLE IF EXISTS {table_name}")
    success, _, nrows, _ = write_pandas(con, df, table_name)
    print(f"{table_name} loaded: {nrows} rows | success={success}")
    if not success:
        raise RuntimeError(f"write_pandas failed for {table_name}")


# ── 1. dim_customers ──
dim_customers = (
    customers[["customer_unique_id", "customer_city", "customer_state"]]
    .drop_duplicates(subset="customer_unique_id")
    .reset_index(drop=True)
)
dim_customers.columns = ["CUSTOMER_UNIQUE_ID", "CUSTOMER_CITY", "CUSTOMER_STATE"]
load_table(dim_customers, "DIM_CUSTOMERS")

# ── 2. dim_products ──
dim_products = products.merge(category_translation, on="product_category_name", how="left")[[
    "product_id", "product_category_name", "product_category_name_english",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"
]].copy()
dim_products.columns = [c.upper() for c in dim_products.columns]
load_table(dim_products, "DIM_PRODUCTS")

# ── 3. dim_sellers ──
dim_sellers = sellers[["seller_id", "seller_city", "seller_state"]].copy()
dim_sellers.columns = ["SELLER_ID", "SELLER_CITY", "SELLER_STATE"]
load_table(dim_sellers, "DIM_SELLERS")

# ── 4. dim_date ──
date_range = pd.date_range(start="2016-01-01", end="2018-12-31", freq="D")
dim_date = pd.DataFrame({
    "DATE_KEY":     date_range.strftime("%Y%m%d").astype(int),
    "FULL_DATE":    date_range.date,
    "YEAR":         date_range.year,
    "QUARTER":      date_range.quarter,
    "MONTH":        date_range.month,
    "MONTH_NAME":   date_range.strftime("%B"),
    "WEEK_OF_YEAR": date_range.isocalendar().week.values,
    "DAY_OF_MONTH": date_range.day,
    "DAY_NAME":     date_range.strftime("%A"),
    "IS_WEEKEND":   date_range.weekday >= 5,
})
load_table(dim_date, "DIM_DATE")

# ── 5. fct_orders ──
payments_agg = payments.groupby("order_id")["payment_value"].sum().reset_index()

reviews_agg = (
    reviews.groupby("order_id")["review_score"]
    .mean()
    .round()
    .astype("Int64")
    .reset_index()
)

fct = (
    order_items
    .merge(
        orders[["order_id", "customer_id", "order_status",
                "order_purchase_timestamp", "order_delivered_customer_date"]],
        on="order_id"
    )
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg,  on="order_id", how="left")
)

# Pull surrogate keys back from Snowflake
dim_cust_sf = cur.execute("SELECT CUSTOMER_KEY, CUSTOMER_UNIQUE_ID FROM DIM_CUSTOMERS").fetch_pandas_all()
dim_cust_sf.columns = dim_cust_sf.columns.str.lower()

dim_prod_sf = cur.execute("SELECT PRODUCT_KEY, PRODUCT_ID FROM DIM_PRODUCTS").fetch_pandas_all()
dim_prod_sf.columns = dim_prod_sf.columns.str.lower()

dim_sell_sf = cur.execute("SELECT SELLER_KEY, SELLER_ID FROM DIM_SELLERS").fetch_pandas_all()
dim_sell_sf.columns = dim_sell_sf.columns.str.lower()

# Map natural keys → surrogate keys
fct = fct.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
fct = fct.merge(dim_cust_sf, on="customer_unique_id", how="left")
fct = fct.merge(dim_prod_sf, on="product_id",          how="left")
fct = fct.merge(dim_sell_sf, on="seller_id",            how="left")

fct["date_key"]         = fct["order_purchase_timestamp"].dt.strftime("%Y%m%d").astype(int)
fct["delivery_days"]    = (fct["order_delivered_customer_date"] - fct["order_purchase_timestamp"]).dt.days
fct["total_item_value"] = fct["price"] + fct["freight_value"]

fct_final = fct[[
    "order_id", "order_item_id", "customer_key", "product_key", "seller_key", "date_key",
    "order_status", "price", "freight_value", "total_item_value", "payment_value",
    "review_score", "order_purchase_timestamp", "order_delivered_customer_date", "delivery_days"
]].copy()
fct_final.columns = [c.upper() for c in fct_final.columns]
load_table(fct_final, "FCT_ORDERS")

# ── Verification ──
verification_sql = """
SELECT 'DIM_CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM DIM_CUSTOMERS
UNION ALL SELECT 'DIM_PRODUCTS',  COUNT(*) FROM DIM_PRODUCTS
UNION ALL SELECT 'DIM_SELLERS',   COUNT(*) FROM DIM_SELLERS
UNION ALL SELECT 'DIM_DATE',      COUNT(*) FROM DIM_DATE
UNION ALL SELECT 'FCT_ORDERS',    COUNT(*) FROM FCT_ORDERS
ORDER BY table_name
"""
result = cur.execute(verification_sql).fetch_pandas_all()
print("\nFinal row counts:")
print(result.to_string(index=False))

cur.close()
con.close()
print("\nConnection closed.")

Connected to Snowflake.
CSVs loaded.
DIM_CUSTOMERS loaded: 96096 rows | success=True
DIM_PRODUCTS loaded: 32951 rows | success=True
DIM_SELLERS loaded: 3095 rows | success=True
DIM_DATE loaded: 1096 rows | success=True
FCT_ORDERS loaded: 112650 rows | success=True

Final row counts:
   TABLE_NAME  ROW_COUNT
DIM_CUSTOMERS      96096
     DIM_DATE       1096
 DIM_PRODUCTS      32951
  DIM_SELLERS       3095
   FCT_ORDERS     112650

Connection closed.


In [3]:
import os
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

if "notebooks" in os.getcwd():
    os.chdir("..")

# ── Snowflake Connection ──
con = snowflake.connector.connect(
    user      = "Ruchita",
    password  = "RuchitaUday@0915",
    account   = "ZGQUOVJ-WT53494",
    warehouse = "COMPUTE_WH",
    database  = "RETAILPULSE_DB",
    schema    = "ANALYTICS"
)
cur = con.cursor()
print("Connected to Snowflake.")

# ── Load raw cleaned CSVs ──
BASE = r"C:\Users\HOLE\DATA ANALYST\RetailPulseAI - Project\01_Data\cleaned"

customers            = pd.read_csv(rf"{BASE}\customers_clean.csv")
products             = pd.read_csv(rf"{BASE}\products_clean.csv")
category_translation = pd.read_csv(rf"{BASE}\category_translation_clean.csv")
sellers              = pd.read_csv(rf"{BASE}\sellers_clean.csv")
orders               = pd.read_csv(rf"{BASE}\orders_clean.csv")
order_items          = pd.read_csv(rf"{BASE}\order_items_clean.csv")
payments             = pd.read_csv(rf"{BASE}\payments_clean.csv")
reviews              = pd.read_csv(rf"{BASE}\reviews_clean.csv")

orders["order_purchase_timestamp"]      = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
print("CSVs loaded.")


def load_table(df, table_name):
    """Truncate then reload a Snowflake table."""
    cur.execute(f"TRUNCATE TABLE IF EXISTS {table_name}")
    success, _, nrows, _ = write_pandas(con, df, table_name)
    print(f"{table_name} loaded: {nrows} rows | success={success}")
    if not success:
        raise RuntimeError(f"write_pandas failed for {table_name}")


# ── 1. dim_customers ──
dim_customers = (
    customers[["customer_unique_id", "customer_city", "customer_state"]]
    .drop_duplicates(subset="customer_unique_id")
    .reset_index(drop=True)
)
dim_customers.columns = ["CUSTOMER_UNIQUE_ID", "CUSTOMER_CITY", "CUSTOMER_STATE"]
load_table(dim_customers, "DIM_CUSTOMERS")

# ── 2. dim_products ──
dim_products = products.merge(category_translation, on="product_category_name", how="left")[[
    "product_id", "product_category_name", "product_category_name_english",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"
]].copy()
dim_products.columns = [c.upper() for c in dim_products.columns]
load_table(dim_products, "DIM_PRODUCTS")

# ── 3. dim_sellers ──
dim_sellers = sellers[["seller_id", "seller_city", "seller_state"]].copy()
dim_sellers.columns = ["SELLER_ID", "SELLER_CITY", "SELLER_STATE"]
load_table(dim_sellers, "DIM_SELLERS")

# ── 4. dim_date ──
date_range = pd.date_range(start="2016-01-01", end="2018-12-31", freq="D")
dim_date = pd.DataFrame({
    "DATE_KEY":     date_range.strftime("%Y%m%d").astype(int),
    "FULL_DATE":    date_range.date,
    "YEAR":         date_range.year,
    "QUARTER":      date_range.quarter,
    "MONTH":        date_range.month,
    "MONTH_NAME":   date_range.strftime("%B"),
    "WEEK_OF_YEAR": date_range.isocalendar().week.values,
    "DAY_OF_MONTH": date_range.day,
    "DAY_NAME":     date_range.strftime("%A"),
    "IS_WEEKEND":   date_range.weekday >= 5,
})
load_table(dim_date, "DIM_DATE")

# ── 5. fct_orders ──
payments_agg = payments.groupby("order_id")["payment_value"].sum().reset_index()

reviews_agg = (
    reviews.groupby("order_id")["review_score"]
    .mean()
    .round()
    .astype("Int64")
    .reset_index()
)

fct = (
    order_items
    .merge(
        orders[["order_id", "customer_id", "order_status",
                "order_purchase_timestamp", "order_delivered_customer_date"]],
        on="order_id"
    )
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg,  on="order_id", how="left")
)

# Pull surrogate keys back from Snowflake
dim_cust_sf = cur.execute("SELECT CUSTOMER_KEY, CUSTOMER_UNIQUE_ID FROM DIM_CUSTOMERS").fetch_pandas_all()
dim_cust_sf.columns = dim_cust_sf.columns.str.lower()

dim_prod_sf = cur.execute("SELECT PRODUCT_KEY, PRODUCT_ID FROM DIM_PRODUCTS").fetch_pandas_all()
dim_prod_sf.columns = dim_prod_sf.columns.str.lower()

dim_sell_sf = cur.execute("SELECT SELLER_KEY, SELLER_ID FROM DIM_SELLERS").fetch_pandas_all()
dim_sell_sf.columns = dim_sell_sf.columns.str.lower()

# Map natural keys → surrogate keys
fct = fct.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
fct = fct.merge(dim_cust_sf, on="customer_unique_id", how="left")
fct = fct.merge(dim_prod_sf, on="product_id",          how="left")
fct = fct.merge(dim_sell_sf, on="seller_id",            how="left")

fct["date_key"]         = fct["order_purchase_timestamp"].dt.strftime("%Y%m%d").astype(int)
fct["delivery_days"]    = (fct["order_delivered_customer_date"] - fct["order_purchase_timestamp"]).dt.days
fct["total_item_value"] = fct["price"] + fct["freight_value"]

fct_final = fct[[
    "order_id", "order_item_id", "customer_key", "product_key", "seller_key", "date_key",
    "order_status", "price", "freight_value", "total_item_value", "payment_value",
    "review_score", "order_purchase_timestamp", "order_delivered_customer_date", "delivery_days"
]].copy()
fct_final.columns = [c.upper() for c in fct_final.columns]
load_table(fct_final, "FCT_ORDERS")

# ── Verification ──
verification_sql = """
SELECT 'DIM_CUSTOMERS' AS table_name, COUNT(*) AS row_count FROM DIM_CUSTOMERS
UNION ALL SELECT 'DIM_PRODUCTS',  COUNT(*) FROM DIM_PRODUCTS
UNION ALL SELECT 'DIM_SELLERS',   COUNT(*) FROM DIM_SELLERS
UNION ALL SELECT 'DIM_DATE',      COUNT(*) FROM DIM_DATE
UNION ALL SELECT 'FCT_ORDERS',    COUNT(*) FROM FCT_ORDERS
ORDER BY table_name
"""
result = cur.execute(verification_sql).fetch_pandas_all()
print("\nFinal row counts:")
print(result.to_string(index=False))

cur.close()
con.close()
print("\nConnection closed.")

Connected to Snowflake.
CSVs loaded.
DIM_CUSTOMERS loaded: 96096 rows | success=True
DIM_PRODUCTS loaded: 32951 rows | success=True
DIM_SELLERS loaded: 3095 rows | success=True
DIM_DATE loaded: 1096 rows | success=True
FCT_ORDERS loaded: 112650 rows | success=True

Final row counts:
   TABLE_NAME  ROW_COUNT
DIM_CUSTOMERS      96096
     DIM_DATE       1096
 DIM_PRODUCTS      32951
  DIM_SELLERS       3095
   FCT_ORDERS     112650

Connection closed.


In [5]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import pandas as pd

con = snowflake.connector.connect(
    user      = "Ruchita",
    password  = "RuchitaUday@0915",
    account   = "ZGQUOVJ-WT53494",
    warehouse = "COMPUTE_WH",
    database  = "RETAILPULSE_DB",
    schema    = "ANALYTICS"
)
cur = con.cursor()

# Create a raw payments table (not aggregated) for payment-method-level analysis
cur.execute("""
CREATE OR REPLACE TABLE raw_payments (
    order_id              VARCHAR(50),
    payment_sequential    NUMBER,
    payment_type          VARCHAR(50),
    payment_installments  NUMBER,
    payment_value         FLOAT
)
""")

payments = pd.read_csv("C:\\Users\\HOLE\\DATA ANALYST\\RetailPulseAI - Project\\01_Data\\cleaned\\payments_clean.csv")
payments.columns = [c.upper() for c in payments.columns]

success, _, nrows, _ = write_pandas(con, payments, "RAW_PAYMENTS")
print(f"raw_payments loaded: {nrows} rows | success={success}")

con.close()

raw_payments loaded: 103886 rows | success=True


In [8]:
import snowflake.connector
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

con = snowflake.connector.connect(
    user="Ruchita", password="RuchitaUday@0915", account="ZGQUOVJ-WT53494",
    warehouse="COMPUTE_WH", database="RETAILPULSE_DB", schema="ANALYTICS"
)
cur = con.cursor()

# Only add column if it doesn't already exist
cur.execute("""
    SELECT COUNT(*) 
    FROM information_schema.columns 
    WHERE table_schema = 'ANALYTICS' 
      AND table_name = 'FCT_ORDERS' 
      AND column_name = 'ORDER_ESTIMATED_DELIVERY_DATE'
""")
col_exists = cur.fetchone()[0]

if not col_exists:
    cur.execute("ALTER TABLE fct_orders ADD COLUMN order_estimated_delivery_date TIMESTAMP_NTZ")
    print("Column added.")
else:
    print("Column already exists, skipping ALTER.")

# Load the estimated delivery dates from cleaned orders
orders = pd.read_csv("C:\\Users\\HOLE\\DATA ANALYST\\RetailPulseAI - Project\\01_Data\\cleaned\\orders_clean.csv",
                     parse_dates=["order_estimated_delivery_date"])
estimate_df = orders[["order_id", "order_estimated_delivery_date"]].copy()
estimate_df.columns = ["ORDER_ID", "ORDER_ESTIMATED_DELIVERY_DATE"]

cur.execute("""
CREATE OR REPLACE TEMPORARY TABLE temp_estimates (
    ORDER_ID VARCHAR, 
    ORDER_ESTIMATED_DELIVERY_DATE TIMESTAMP_NTZ
)
""")
write_pandas(con, estimate_df, "TEMP_ESTIMATES")

cur.execute("""
UPDATE fct_orders f
SET order_estimated_delivery_date = t.order_estimated_delivery_date
FROM temp_estimates t
WHERE f.order_id = t.order_id
""")
print("order_estimated_delivery_date populated.")
con.close()

Column already exists, skipping ALTER.
order_estimated_delivery_date populated.


In [10]:
import snowflake.connector
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

con = snowflake.connector.connect(
    user="Ruchita", password="RuchitaUday@0915", account="ZGQUOVJ-WT53494",
    warehouse="COMPUTE_WH", database="RETAILPULSE_DB", schema="ANALYTICS"
)
cur = con.cursor()

# Create a raw reviews table for review-text-level analysis
cur.execute("""
CREATE OR REPLACE TABLE raw_reviews (
    review_id                  VARCHAR(50),
    order_id                   VARCHAR(50),
    review_score                NUMBER,
    review_comment_title        VARCHAR(255),
    review_comment_message      VARCHAR(5000),
    review_creation_date         TIMESTAMP_NTZ,
    review_answer_timestamp      TIMESTAMP_NTZ
)
""")

reviews = pd.read_csv("C:\\Users\\HOLE\\DATA ANALYST\\RetailPulseAI - Project\\01_Data\\cleaned\\reviews_clean.csv",
                      parse_dates=["review_creation_date", "review_answer_timestamp"])
reviews.columns = [c.upper() for c in reviews.columns]

success, _, nrows, _ = write_pandas(con, reviews, "RAW_REVIEWS")
print(f"raw_reviews loaded: {nrows} rows | success={success}")

con.close()

raw_reviews loaded: 99224 rows | success=True


In [12]:
import pandas as pd

orders = pd.read_csv(
    "C:\\Users\\HOLE\\DATA ANALYST\\RetailPulseAI - Project\\01_Data\\cleaned\\orders_clean.csv",
    parse_dates=["order_purchase_timestamp"]
)

print("Earliest Order Date :", orders["order_purchase_timestamp"].min())
print("Latest Order Date   :", orders["order_purchase_timestamp"].max())
print("Missing Values      :", orders["order_purchase_timestamp"].isnull().sum())

Earliest Order Date : 2016-09-04 21:15:19
Latest Order Date   : 2018-10-17 17:30:18
Missing Values      : 0


In [14]:
import snowflake.connector
import pandas as pd
from snowflake.connector.pandas_tools import write_pandas

con = snowflake.connector.connect(
    user="Ruchita", password="RuchitaUday@0915", account="ZGQUOVJ-WT53494",
    warehouse="COMPUTE_WH", database="RETAILPULSE_DB", schema="ANALYTICS"
)
cur = con.cursor()

# Re-load orders with explicit, verified-clean datetime parsing
orders = pd.read_csv("C:\\Users\\HOLE\\DATA ANALYST\\RetailPulseAI - Project\\01_Data\\cleaned\\orders_clean.csv")
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

# Sanity check BEFORE writing anything to Snowflake — this is the critical checkpoint
print("Min:", orders["order_purchase_timestamp"].min())
print("Max:", orders["order_purchase_timestamp"].max())
print("Nulls:", orders["order_purchase_timestamp"].isnull().sum())
print("Total rows:", len(orders))

Min: 2016-09-04 21:15:19
Max: 2018-10-17 17:30:18
Nulls: 0
Total rows: 99441


In [15]:
# Convert to plain ISO string format - the most reliable way to hand dates to Snowflake
orders["order_purchase_timestamp_str"] = orders["order_purchase_timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Stage just order_id + corrected timestamp string
fix_df = orders[["order_id", "order_purchase_timestamp_str"]].copy()
fix_df.columns = ["ORDER_ID", "FIXED_TIMESTAMP_STR"]

cur.execute("CREATE OR REPLACE TEMPORARY TABLE temp_fix (ORDER_ID VARCHAR, FIXED_TIMESTAMP_STR VARCHAR)")
write_pandas(con, fix_df, "TEMP_FIX")

# Update fct_orders using an explicit TO_TIMESTAMP conversion from clean text
cur.execute("""
UPDATE fct_orders f
SET order_purchase_timestamp = TO_TIMESTAMP(t.FIXED_TIMESTAMP_STR, 'YYYY-MM-DD HH24:MI:SS')
FROM temp_fix t
WHERE f.order_id = t.order_id
""")

print("order_purchase_timestamp repaired.")

# Immediately verify
check = cur.execute("SELECT MIN(order_purchase_timestamp), MAX(order_purchase_timestamp) FROM fct_orders").fetchone()
print("New min/max:", check)

con.close()

order_purchase_timestamp repaired.
New min/max: (datetime.datetime(2016, 9, 4, 21, 15, 19), datetime.datetime(2018, 9, 3, 9, 6, 57))
